In [ ]:

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "6"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["HF_TOKEN"] = ""
os.environ["HF_HOME"] = "/gpfs/home/exaone/.cache/huggingface/"

from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Qwen3-14B
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-14B")

from datasets import load_dataset

ds_lst = []
ds_lst.extend(load_dataset("eungizoa/fsec-dacon-250826", name="SFT-OEQA", split="법령외").to_list())
ds_lst.extend(load_dataset("eungizoa/fsec-dacon-250826", name="SFT-OEQA", split="국가법령정보센터").to_list())
ds_lst.extend(load_dataset("eungizoa/fsec-dacon-250826", name="SFT-MCQA", split="법령외").to_list())
ds_lst.extend(load_dataset("eungizoa/fsec-dacon-250826", name="SFT-MCQA", split="국가법령정보센터").to_list())

In [3]:
len(ds_lst)

37199

In [ ]:
from alignment.prompts import (
    MCQA_SYSTEM_PROMPT, MCQA_INSTRUCTION_TEMPLATE, MCQA_ANSWER_TEMPLATE, MCQA_ANSWER_COT_TEMPLATE,
    OEQA_SYSTEM_PROMPT, OEQA_INSTRUCTION_TEMPLATE, OEQA_ANSWER_TEMPLATE, OEQA_ANSWER_COT_TEMPLATE
)

def format_messages(example: dict):
        
    # Public instruction-following training dataset generally has `messages` field
    if example.get("messages"):
        return example
    # Custom datasets require messages formatting
    else:            
        is_mcqa = example.get("options", None) is not None
        if is_mcqa:
            system_message = MCQA_SYSTEM_PROMPT
            options_str = "\n".join([f"{i+1}. {option}" for i, option in enumerate(example["options"])])
            user_message = MCQA_INSTRUCTION_TEMPLATE.format(query=example["query"], options=options_str)
            if (cot_trace := example["cot_trace"].strip()) != "":
                assistant_message = MCQA_ANSWER_COT_TEMPLATE.format(trace=cot_trace, answer=example["answer"])
            else:
                assistant_message = MCQA_ANSWER_TEMPLATE.format(answer=example["answer"])
        else:
            system_message = OEQA_SYSTEM_PROMPT
            user_message = OEQA_INSTRUCTION_TEMPLATE.format(query=example["question"])
            assistant_message = OEQA_ANSWER_TEMPLATE.format(answer=example["answer"])
            if (cot_trace := example["cot_trace"].strip()) != "":
                assistant_message = OEQA_ANSWER_COT_TEMPLATE.format(trace=cot_trace, answer=example["answer"])
            else:
                assistant_message = OEQA_ANSWER_TEMPLATE.format(answer=example["answer"])
        
        # # assistant-only loss
        example["messages"] = [
            {"role": "system", "content": system_message}, {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]
        
        # Format chat_template_kwargs
        chat_template_kwargs = example.get("chat_template_kwargs") or {}
        chat_template_kwargs.update({"enable_thinking": True})
        example["chat_template_kwargs"] = chat_template_kwargs # it will be feed into the `apply_chat_template` function
    return example

_ds_lst = []
for sample in ds_lst:
    _ds_lst += [format_messages(sample)]
ds_lst = _ds_lst
del _ds_lst

In [ ]:
num_tokens = 0
for sample in ds_lst:
    inputs = tokenizer.apply_chat_template(sample["messages"], tokenize=True, return_dict=True, add_generation_prompt=False, **sample["chat_template_kwargs"])
    num_tokens += len(inputs["input_ids"])

In [ ]:
print(tokenizer.decode(inputs["input_ids"]))

In [ ]:
num_tokens

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "6"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
from tqdm import tqdm
from alignment.prompts import OEQA_INSTRUCTION_TEMPLATE, OEQA_SYSTEM_PROMPT, MCQA_SYSTEM_PROMPT, MCQA_INSTRUCTION_TEMPLATE


# Qwen3-14B
tokenizer = AutoTokenizer.from_pretrained("/gpfs/home/eungizoa/models/doctrio/qwen3-14b-mcqa-oeqa-250828/checkpoint-150")
model = AutoModelForCausalLM.from_pretrained("/gpfs/home/eungizoa/models/doctrio/qwen3-14b-mcqa-oeqa-250828/checkpoint-150", torch_dtype=torch.bfloat16).to("cuda:0")

raw_datasets = pd.read_csv("/gpfs/home/eungizoa/data/hydra-fineval/preliminary/test.parsed.csv").to_dict(orient="records")
for example in raw_datasets:
    example["options"] = eval(example["options"])
    if example["options"]:
        system_message = MCQA_SYSTEM_PROMPT
        options_str = "\n".join([f"{i+1}. {option}" for i, option in enumerate(example["options"])])
        user_message = MCQA_INSTRUCTION_TEMPLATE.format(query=example["query"], options=options_str)
    else:
        system_message = OEQA_SYSTEM_PROMPT
        user_message = OEQA_INSTRUCTION_TEMPLATE.format(query=example["query"])
        
    example["messages"] = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [2]:
print(raw_datasets[0]["messages"][-1]["content"])

Provide a detailed explanation to solve the question if possible. End your response with the correct option number in the format "Answer: $OPTION (an integer number)".
Question: 금융산업의 이해와 관련하여 금융투자업의 구분에 해당하지 않는 것은?
Options: 
1. 소비자금융업
2. 투자자문업
3. 투자매매업
4. 투자중개업
5. 보험중개업


In [3]:
# Qwen3-14B
model.eval()

with torch.no_grad():
    for example in tqdm(raw_datasets):
        model_inputs = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=True,
            return_tensors="pt",
            return_dict=True
        )
        outputs = model.generate(**model_inputs.to(model.device), max_new_tokens=1024)
        response_ids = outputs[0][len(model_inputs["input_ids"][0]):].tolist()
        response = tokenizer.decode(response_ids, skip_special_tokens=True)
        example["response"] = response

100%|██████████| 515/515 [1:25:48<00:00, 10.00s/it]


In [4]:
pd.DataFrame(raw_datasets).iloc[:, 1:].to_csv("/gpfs/home/eungizoa/data/hydra-fineval/submission/qwen-250828-13pm.csv", index=None, encoding="utf-8-sig")

In [5]:
import os
import re

def extract_cot_and_final_answer(text):
    """
    Extract thinking traces and final answer from text.
    
    Returns:
        tuple: (thinking_trace, answer)
    """
    # Pattern to match thinking traces (everything before "Answer:")
    thinking_pattern = r"<think>(.*?)</think>"
    
    # Pattern to match the answer after "Answer:"
    answer_pattern = r'.*Answer:\s*(.*)$'
    
    thinking_match = re.search(thinking_pattern, text, re.DOTALL | re.MULTILINE)
    answer_match = list(re.finditer(answer_pattern, text, re.DOTALL | re.MULTILINE))
    
    if answer_match:
        answer_match = answer_match[-1]
        cleaned_answer = answer_match.group(1).strip()
    else:
        cleaned_answer = text
    
    if thinking_match:
        thinking_trace = thinking_match.group(1).strip()
        if cleaned_answer == text:
            cleaned_answer = re.sub(thinking_pattern, "", cleaned_answer, flags=re.DOTALL)
    else:
        thinking_trace = text
        cleaned_answer = "0"
    
    return thinking_trace.strip(), cleaned_answer.strip()

for example in raw_datasets:
    cot_trace, answer = extract_cot_and_final_answer(example["response"])
    example["is_mcqa"] = True if example["options"] else False
    if example["is_mcqa"]:
        example["parsed_cot_trace"] = cot_trace
        example["parsed_answer"] = answer if answer else "0"
    else:
        example["parsed_cot_trace"] = cot_trace
        example["parsed_answer"] = answer

In [6]:
submission = pd.DataFrame([{"ID": example["ID"], "Answer": example["parsed_answer"], "is_mcqa": example["is_mcqa"]} for example in raw_datasets])

In [7]:
submission[submission["is_mcqa"] == True]["Answer"].value_counts()

Answer
3    126
4    115
1    109
2     97
5     49
0      3
6      1
Name: count, dtype: int64

In [8]:
submission[(submission["is_mcqa"] == True) & (submission["Answer"] == "6")]

,ID,Answer,is_mcqa
164,TEST_164,6,True


In [9]:
[example for example in raw_datasets if example["ID"] == "TEST_164"][0]

{'Unnamed: 0': 164,
 'ID': 'TEST_164',
 'Question': '다음 중 보안 취약점에 의한 정보유출을 방지하기 위한 기술적 보안 대책은 무엇인가?\n1 보안 정책 수립\n2 접근 통제 강화\n3 내부 감사 실시\n4 보안 교육 강화\n5 위험 평가 보고서 작성',
 'query': '다음 중 보안 취약점에 의한 정보유출을 방지하기 위한 기술적 보안 대책은 무엇인가?',
 'options': ['보안 정책 수립', '접근 통제 강화', '내부 감사 실시', '보안 교육 강화', '위험 평가 보고서 작성'],
 'messages': [{'role': 'system',
   'content': 'You are a knowledgeable assistant that answers multiple-choice questions. Choose the correct option based on the question and options provided. Explain your reasoning if possible.'},
  {'role': 'user',
   'content': 'Provide a detailed explanation to solve the question if possible. End your response with the correct option number in the format "Answer: $OPTION (an integer number)".\nQuestion: 다음 중 보안 취약점에 의한 정보유출을 방지하기 위한 기술적 보안 대책은 무엇인가?\nOptions: \n1. 보안 정책 수립\n2. 접근 통제 강화\n3. 내부 감사 실시\n4. 보안 교육 강화\n5. 위험 평가 보고서 작성'}],
 'response': '<think>\n정보유출을 방지하기 위한 기술적 보안 대책은 보안 정책 수립, 접근 통제 강화, 내부 감사 실시, 보안 교육 강화, 위험 평가 보고서 작성 등의 관리적 대책과 병행하여 적용된다

In [10]:
submission.drop(columns=["is_mcqa"]).to_csv("/gpfs/home/eungizoa/data/hydra-fineval/submission/submission-250828-13pm.csv", encoding="utf-8-sig", index=None)